# QAOA angle prediction — models loaded from Hugging Face

Downloads released QAOA angle-prediction models straight from the Hub (no local
checkpoint tree needed), predicts `[β₁..β_p, γ₁..γ_p]` for a graph, and optionally
scores the angles by MaxCut approximation ratio.

Each Hub repo is a self-contained ONNX bundle — `model_config.json`, `model.onnx`,
`model.onnx.data` — named `ibm-research/qaoa_angles.max_cut.p{p}.{model}`.

**Prerequisites**
- the training-pipeline venv: `../qaoa_training_pipeline/venv/bin/python -m ipykernel install --user`
- `pip install huggingface_hub`
- the repos are private, so authenticate once: `hf auth login` (or export `HF_TOKEN`)

## 1. Configure

In [1]:
# Hub repos to run: "label" -> repo id.
# Naming scheme: ibm-research/qaoa_angles.max_cut.p{p}.{model}
models = [
    "graph_neural_network/p1"
]

# Graph to predict angles for (any networkx graph works — see cell 5).
graph_spec = ("random_regular", {"d": 3, "n": 36, "seed": 7})

## 2. Setup

In [2]:
from __future__ import annotations

try:
    import juliacall  # noqa: F401 — must precede torch (pytorch#78829)
except ImportError:
    pass

import sys
from pathlib import Path

import networkx as nx
import pandas as pd

from qaoa_training_pipeline.inference import AIInference
from qaoa_training_pipeline.utils.graph_utils import graph_to_operator

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

maxcut_pre_factor = -0.5  # training convention: H = -0.5 * sum_e w_e Z_i Z_j

print(f"[setup] repo_root={repo_root}")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
[setup] repo_root=/Users/jja/Documents/04_Quantum/qaoa_training_pipeline_public/how_tos


## 4. Build the graph and its cost operator

In [3]:
graph_builders = {
    "random_regular": lambda d, n, seed: nx.random_regular_graph(d, n, seed=seed),
    "erdos_renyi":    lambda n, prob, seed: nx.erdos_renyi_graph(n, prob, seed=seed),
    "barabasi_albert": lambda n, m, seed: nx.barabasi_albert_graph(n, m, seed=seed),
}

kind, kwargs = graph_spec
graph = graph_builders[kind](**kwargs)
nx.set_edge_attributes(graph, 1.0, "weight")  # unweighted MaxCut

cost_op = graph_to_operator(graph, pre_factor=maxcut_pre_factor)

print(f"[graph] {kind} {kwargs} -> {graph.number_of_nodes()} nodes, "
      f"{graph.number_of_edges()} edges, {cost_op.num_qubits} qubits")

[graph] random_regular {'d': 3, 'n': 36, 'seed': 7} -> 36 nodes, 54 edges, 36 qubits


## 5. Predict angles (+ approximation ratio)

In [4]:
def approx_ratio(cost_op, angles: list[float]) -> float:
    from qaoa_training_pipeline.evaluation import EVALUATORS
    from qaoa_training_pipeline.utils.graph_utils import solve_max_cut

    energy = float(EVALUATORS["PPEvaluator"]().evaluate(cost_op, angles))
    _max_cut, _min_cut, ratio = solve_max_cut(cost_op, energy=energy)
    return float(ratio)


rows = []

for model in models:

    ai = AIInference(model=model)
    angles = ai.provide_params(cost_op)["optimized_params"]
    p = int(model[-1])

    rows.append({
        "model": model,
        "num_qubits": cost_op.num_qubits,
        "betas": [round(x, 4) for x in angles[:p]],
        "gammas": [round(x, 4) for x in angles[p:]],
        "approx_ratio": round(approx_ratio(cost_op, angles), 4),
    })

results = pd.DataFrame(rows)
results

/Users/jja/Documents/04_Quantum/qaoa_training_pipeline/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 935.18it/s]


,model,num_qubits,betas,gammas,approx_ratio
0,graph_neural_network/p1,36,[0.3876],[0.6096],0.7531


## 6. Use it on your own graph

Everything above reduces to three lines — the only input a model needs is a
`SparsePauliOp` cost operator.

In [5]:
my_graph = nx.erdos_renyi_graph(14, 0.3, seed=1)
nx.set_edge_attributes(my_graph, 1.0, "weight")

cost_op = graph_to_operator(my_graph, pre_factor=maxcut_pre_factor)

ai = AIInference(model=model)
angles = ai.provide_params(cost_op)["optimized_params"]

print("beta =", angles[:len(angles) // 2], " gamma =", angles[len(angles) // 2:])
print("Approximation ratio: ", round(approx_ratio(cost_op, angles), 4))

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 1948.12it/s]

beta = [0.3445764482021332]  gamma = [0.4999809265136719]


Approximation ratio:  0.8122
